# 00 Environment and Data Check

Проверка локальных файлов Thebe и базовой структуры данных.

Базовые классы и функции берутся из `src/seismic_fault_recognition`. Данные должны уже лежать локально; скачивания в ноутбуке нет.

## 1. Bootstrap

Добавляем `src` в `sys.path`, чтобы ноутбук запускался прямо из репозитория.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

## 2. Конфигурация эксперимента

Загружаем recipe и YAML-конфиг. Все пути редактируются в `configs/experiments`, а не в коде ноутбука.

In [ ]:
RECIPE_NAME = "environment_and_data_check"
from seismic_fault_recognition.notebook_utils import (
    ensure_dir,
    load_notebook_context,
    print_context_summary,
    print_path_report,
    seed_everything,
    tuple3,
)

ctx = load_notebook_context(RECIPE_NAME, start=repo_root)
DATA = ctx.data
TRAINING = ctx.training
CHECKPOINTS = ctx.checkpoints
OUTPUTS = ctx.outputs

seed_everything(
    int(ctx.reproducibility.get("seed", TRAINING.get("seed", 42))),
    deterministic=bool(ctx.reproducibility.get("deterministic", True)),
    benchmark=bool(ctx.reproducibility.get("benchmark", False)),
)
print_context_summary(ctx)

## ClearML logging

Создаем ClearML task из recipe/config. Если `clearml` не установлен или отключен в конфиге, объект будет работать как no-op и обучение продолжится локально.

In [ ]:
from seismic_fault_recognition.clearml import (
    clearml_metric_logger,
    init_clearml_from_context,
    report_checkpoint_artifact,
    report_optimizer_lr,
)

clearml_run = init_clearml_from_context(ctx)
print(clearml_run.summary())

## 3. Быстрая проверка путей

Эта ячейка ничего не создает и не скачивает; она только показывает, какие локальные файлы уже доступны.

In [ ]:
print_path_report(
    {
        "train_seis": DATA.get("train_seis", ""),
        "train_fault": DATA.get("train_fault", ""),
        "val_seis": DATA.get("val_seis", ""),
        "val_fault": DATA.get("val_fault", ""),
        "test_seis": DATA.get("test_seis", ""),
        "test_fault": DATA.get("test_fault", ""),
        "output_dir": CHECKPOINTS.get("output_dir", ""),
    },
    ctx.repo_root,
)

## 4. Импорты

In [ ]:
from seismic_fault_recognition.data import analyze_npz, require_paths

## 5. Проверяем обязательные локальные файлы

In [ ]:
paths = require_paths(
    {
        "train_seis": DATA["train_seis"],
        "train_fault": DATA["train_fault"],
        "val_seis": DATA["val_seis"],
        "val_fault": DATA["val_fault"],
    },
    base_dir=ctx.repo_root,
)

## 6. Краткая сводка NPZ

In [ ]:
for name, path in paths.items():
    print(f"\n{name}: {path}")
    for row in analyze_npz(path)[:10]:
        print(row)